# Egyptian National ID OCR in Google Colab

This notebook clones the repository, installs the dependencies, uploads an Egyptian National ID image, runs the OCR pipeline, visualizes the preprocessing stages, and exports the extracted result as JSON.

Target fields:

- Full Name (Arabic)
- Address (Arabic)
- National ID Number (14 digits)


## Notebook Workflow

1. Clone the GitHub repository into Colab.
2. Install Python dependencies.
3. Upload an image from your machine.
4. Run preprocessing and OCR.
5. Review extracted fields and confidence.
6. Save annotated output and JSON result.


In [ ]:
import os

REPO_URL = "https://github.com/gawadx1/masrid-ocr.git"
REPO_DIR = "/content/masrid-ocr"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/masrid-ocr


In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt


In [ ]:
# Optional: set this to True if you want to read images from Google Drive.
USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if not uploaded:
    raise ValueError("No image uploaded.")

image_name = next(iter(uploaded))
image_path = str(Path('/content') / image_name)
print(f"Using image: {image_path}")


In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

from app.ocr import get_ocr_service
from app.postprocess import build_extraction_response
from app.preprocess import preprocess_image
from app.utils import draw_bounding_boxes, ensure_directory, export_results_to_json

artifacts = preprocess_image(image_path)
tokens = get_ocr_service().extract_tokens(artifacts['card'])
result = build_extraction_response(tokens)

output_dir = ensure_directory('outputs')
stem = Path(image_path).stem
annotated_path = output_dir / f"{stem}_annotated.jpg"
json_path = output_dir / f"{stem}_result.json"

annotated = draw_bounding_boxes(artifacts['card'], tokens, annotated_path)
export_results_to_json(result.model_dump(), json_path)

print(json.dumps(result.model_dump(), ensure_ascii=False, indent=2))
print(f"Annotated image saved to: {annotated_path}")
print(f"JSON result saved to: {json_path}")


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 6))

axes[0].imshow(cv2.cvtColor(artifacts['original'], cv2.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(artifacts['card'], cv2.COLOR_BGR2RGB))
axes[1].set_title('Detected Card')
axes[1].axis('off')

axes[2].imshow(artifacts['thresholded'], cmap='gray')
axes[2].set_title('Thresholded')
axes[2].axis('off')

axes[3].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[3].set_title('OCR Boxes')
axes[3].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
for index, token in enumerate(tokens, start=1):
    print(f"{index:02d} | conf={token.confidence:.3f} | text={token.text}")


## Next Steps

- Try multiple mobile photos with different lighting conditions.
- Compare extracted values against ground truth for your dataset.
- Export the `outputs/` folder to Drive if you want to keep the annotated results.
- Move from notebook experimentation to the FastAPI service for production integration.
